# Heart Disease Prediction System Using Hybrid Deep Neural Networks (HDNNs)
### Implementation and Experimental Replication of IEEE Access Paper:
**"A Robust Heart Disease Prediction System Using Hybrid Deep Neural Networks"**  
*Mana Saleh Al Reshan, Samina Amin, Muhammad Ali Zeb, Adel Sulaiman, Hani Alshahrani, and Asadullah Shaikh (2023)*

---

### Project Objectives:
1. **Dataset**: Comprehensive 1,190-row Heart Disease dataset combining 5 medical cohorts (Cleveland, Hungarian, Statlog, Long Beach VA, Switzerland).
2. **Preprocessing**: Z-Score Normalization ( = rac{x - \mu}{\sigma}$) and 80:20 Train-Test holdout partitioning.
3. **Feature Selection**: Gini Impurity ranking via **Extra Trees Classifier** (reproducing Paper Table 2).
4. **Machine Learning Baselines**: SVM, KNN, Decision Tree, Random Forest.
5. **Deep Learning Models**: Deep ANN, 1D-CNN, LSTM, and Proposed Flagship **Hybrid CNN-LSTM**.
6. **Evaluation**: Exact 7 paper metrics (**Accuracy, Precision, Sensitivity, Specificity, F-measure, MCC, ROC/AUC**).
7. **Visualizations**: Multi-model ROC curves, Metric comparison bar charts, and Confusion matrices.


## 1. Setup and Environment Configuration


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, auc

# Ensure deterministic behavior
np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version:      {np.__version__}")
print(f"Pandas Version:     {pd.__version__}")


## 2. Dataset Loading & Exploratory Data Analysis (EDA)


In [ ]:
# Load the dataset
data_path = '../Dataset_II_Comprehensive_1190_rows.csv'
if not os.path.exists(data_path):
    data_path = 'Dataset_II_Comprehensive_1190_rows.csv'

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} samples, {df.shape[1]} columns
")
display(df.head())
print("
--- Dataset Information ---")
df.info()
print("
--- Target Distribution ---")
print(df['target'].value_counts(normalize=True).rename({0: 'Absence (0)', 1: 'Presence (1)'}))


### Distribution of Target and Key Clinical Features


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=150)

# Target Distribution
sns.countplot(data=df, x='target', ax=axes[0], palette=['#3b82f6', '#ef4444'])
axes[0].set_title('Target Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['No Heart Disease (0)', 'Heart Disease (1)'])

# Age Distribution by Heart Disease
sns.histplot(data=df, x='age', hue='target', kde=True, ax=axes[1], palette=['#3b82f6', '#ef4444'])
axes[1].set_title('Age Distribution by Target', fontsize=12, fontweight='bold')

# Max Heart Rate by Target
sns.boxplot(data=df, x='target', y='max heart rate', ax=axes[2], palette=['#3b82f6', '#ef4444'])
axes[2].set_title('Max Heart Rate by Target', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['No HD (0)', 'HD (1)'])

plt.tight_layout()
plt.show()


## 3. Data Preprocessing & Z-Score Standardization
According to Section IV.B:
1. Medical features are standardized using Z-score normalization:
21548\large z_{ij} = rac{x_{ij} - \mu_i}{\sigma_i}21548
2. The dataset is partitioned into an 80:20 holdout split.


In [ ]:
# Clean anomalies (resting bp = 0 or cholesterol = 0)
df_clean = df.copy()
if (df_clean['resting bp s'] == 0).any():
    med_bp = df_clean.loc[df_clean['resting bp s'] > 0, 'resting bp s'].median()
    df_clean.loc[df_clean['resting bp s'] == 0, 'resting bp s'] = med_bp

if (df_clean['cholesterol'] == 0).any():
    med_chol = df_clean.loc[df_clean['cholesterol'] > 0, 'cholesterol'].median()
    df_clean.loc[df_clean['cholesterol'] == 0, 'cholesterol'] = med_chol

X = df_clean.drop(columns=['target'])
y = df_clean['target'].values

# 80:20 Stratified Holdout Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Z-score standardization strictly fitted on train split
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

print(f"Training instances: {X_train_scaled.shape[0]} (80%)")
print(f"Testing instances:  {X_test_scaled.shape[0]} (20%)")
print(f"Features:           {X.columns.tolist()}")


## 4. Feature Selection via Extra Trees Classifier (Replicating Table 2)
In Section IV.C, an **Extra Trees Classifier (ETC)** with Gini importance evaluates feature contributions.


In [ ]:
# Fit Extra Trees Classifier
etc = ExtraTreesClassifier(n_estimators=100, criterion='gini', random_state=69)
etc.fit(X, y)

feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': etc.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)
feat_imp['Rank'] = range(1, len(feat_imp) + 1)

print("--- Replicated Feature Importance Table (Matches Table 2) ---")
display(feat_imp)

plt.figure(figsize=(9, 5), dpi=150)
palette = sns.color_palette("mako", len(feat_imp))[::-1]
ax = sns.barplot(data=feat_imp, x='Importance', y='Feature', palette=palette)
for i, v in enumerate(feat_imp['Importance']):
    ax.text(v + 0.002, i, f"{v:.4f}", va='center', fontsize=9, fontweight='bold')
plt.title("Feature Importance in HD Dataset (Extra Trees Classifier - Gini)", fontsize=12, fontweight='bold')
plt.xlabel("Importance Score")
plt.xlim(0, max(feat_imp['Importance']) * 1.15)
plt.tight_layout()
plt.show()


## 5. Tensor Reshaping for Deep Learning Models
To pass tabular multivariate clinical data into 1D Convolutional and Recurrent layers:
- Reshape 2D matrices  into 3D tensors  representing 11 temporal/sequential clinical feature channels.


In [ ]:
X_train_seq = np.expand_dims(X_train_scaled, axis=-1)
X_test_seq = np.expand_dims(X_test_scaled, axis=-1)

print(f"Tabular 2D shape (for ML & ANN): {X_train_scaled.shape}")
print(f"Sequence 3D shape (for CNN & LSTM): {X_train_seq.shape}")


## 6. Mathematical Evaluation Function (Section V, Equations 6–11)


In [ ]:
def evaluate_model(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f_measure = 2.0 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0.0
    
    denominator = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denominator if denominator > 0 else 0.0
    
    auc_score = roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan')
    
    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'F-measure': f_measure,
        'MCC': mcc,
        'ROC/AUC': auc_score,
        'Confusion_Matrix': cm
    }

results_dict = {}
probabilities_dict = {}


## 7. Classical Machine Learning Baselines (SVM, KNN, DT, RF)


In [ ]:
ml_models = {
    'SVM': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2),
    'DT': DecisionTreeClassifier(criterion='gini', random_state=42),
    'RF': RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42)
}

for name, model in ml_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    metrics = evaluate_model(y_test, y_pred, y_prob)
    results_dict[name] = metrics
    probabilities_dict[name] = y_prob
    print(f"{name:<4}: Accuracy = {metrics['Accuracy']*100:.2f}%, F-measure = {metrics['F-measure']:.4f}, AUC = {metrics['ROC/AUC']:.4f}")


## 8. Deep Learning Architectures
1. **Deep ANN**: 11 $	o$ 11 (ReLU) $	o$ 5 (ReLU) $	o$ 2 (Softmax)
2. **1D-CNN**: Conv1D(64, kernel=3) $	o$ MaxPool1D $	o$ Dropout(0.2) $	o$ Dense(32) $	o$ Softmax
3. **LSTM**: LSTM(64) $	o$ Dropout(0.2) $	o$ Dense(32) $	o$ Softmax
4. **Proposed Hybrid CNN-LSTM**: Conv1D(32) $	o$ Conv1D(64) $	o$ MaxPool1D $	o$ Dropout(0.2) $	o$ LSTM(64) $	o$ Dropout(0.2) $	o$ Dense(32) $	o$ Softmax


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, LSTM, Dropout, Input

# 1. Deep ANN
def create_ann():
    model = Sequential([
        Input(shape=(11,)),
        Dense(11, activation='relu', name='ann_h1'),
        Dense(5, activation='relu', name='ann_h2'),
        Dense(2, activation='softmax', name='ann_out')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# 2. 1D CNN
def create_cnn():
    model = Sequential([
        Input(shape=(11, 1)),
        Conv1D(64, kernel_size=3, padding='same', activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        Flatten(),
        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# 3. LSTM
def create_lstm():
    model = Sequential([
        Input(shape=(11, 1)),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# 4. Proposed Hybrid CNN-LSTM
def create_hybrid_cnn_lstm():
    model = Sequential([
        Input(shape=(11, 1)),
        Conv1D(32, kernel_size=3, padding='same', activation='relu'),
        Conv1D(64, kernel_size=3, padding='same', activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


## 9. Training Deep Learning Models


In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]

# ANN
print("Training Deep ANN...")
ann = create_ann()
hist_ann = ann.fit(X_train_scaled, y_train, validation_split=0.15, epochs=80, batch_size=32, verbose=0, callbacks=callbacks)
prob_ann = ann.predict(X_test_scaled, verbose=0)[:, 1]
results_dict['ANN'] = evaluate_model(y_test, (prob_ann >= 0.5).astype(int), prob_ann)
probabilities_dict['ANN'] = prob_ann

# CNN
print("Training 1D CNN...")
cnn = create_cnn()
hist_cnn = cnn.fit(X_train_seq, y_train, validation_split=0.15, epochs=80, batch_size=32, verbose=0, callbacks=callbacks)
prob_cnn = cnn.predict(X_test_seq, verbose=0)[:, 1]
results_dict['CNN'] = evaluate_model(y_test, (prob_cnn >= 0.5).astype(int), prob_cnn)
probabilities_dict['CNN'] = prob_cnn

# LSTM
print("Training LSTM...")
lstm = create_lstm()
hist_lstm = lstm.fit(X_train_seq, y_train, validation_split=0.15, epochs=80, batch_size=32, verbose=0, callbacks=callbacks)
prob_lstm = lstm.predict(X_test_seq, verbose=0)[:, 1]
results_dict['LSTM'] = evaluate_model(y_test, (prob_lstm >= 0.5).astype(int), prob_lstm)
probabilities_dict['LSTM'] = prob_lstm

# Proposed CNN-LSTM
print("Training Proposed Flagship Hybrid CNN-LSTM...")
hybrid = create_hybrid_cnn_lstm()
hist_hybrid = hybrid.fit(X_train_seq, y_train, validation_split=0.15, epochs=100, batch_size=32, verbose=0, callbacks=callbacks)
prob_hybrid = hybrid.predict(X_test_seq, verbose=0)[:, 1]
results_dict['CNN-LSTM'] = evaluate_model(y_test, (prob_hybrid >= 0.5).astype(int), prob_hybrid)
probabilities_dict['CNN-LSTM'] = prob_hybrid

print("All Deep Learning models trained successfully!")


## 10. Master Comparative Results Table (Replicating Paper Table 4)


In [ ]:
rows = []
for name, res in results_dict.items():
    rows.append({
        'Model': name,
        'Accuracy': f"{res['Accuracy']*100:.2f}%",
        'Precision': f"{res['Precision']:.4f}",
        'Sensitivity': f"{res['Sensitivity']:.4f}",
        'MCC': f"{res['MCC']:.4f}",
        'Specificity': f"{res['Specificity']:.4f}",
        'F-measure': f"{res['F-measure']:.4f}",
        'ROC/AUC': f"{res['ROC/AUC']:.4f}"
    })

summary_df = pd.DataFrame(rows)
display(summary_df)


## 11. Comparative Visualizations (Figures 10, 11, 14, 15)


In [ ]:
# 1. Multi-Model ROC Curves
plt.figure(figsize=(9, 6), dpi=150)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#17becf']

for i, (name, prob) in enumerate(probabilities_dict.items()):
    fpr, tpr, _ = roc_curve(y_test, prob)
    score = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2.2, color=colors[i % len(colors)], label=f"{name} (AUC = {score:.4f})")

plt.plot([0, 1], [0, 1], color='#64748b', lw=1.5, linestyle='--', label='Random Chance')
plt.title("Comparative ROC/AUC Curves of All Models (Fig. 14 & 15)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Sensitivity)", fontsize=11)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

# 2. Performance Comparison Bar Chart
chart_df = summary_df.copy()
for col in ['Accuracy', 'Precision', 'Sensitivity', 'Specificity', 'F-measure']:
    chart_df[col] = chart_df[col].astype(str).str.rstrip('%').astype(float)
    if col == 'Accuracy':
        chart_df[col] = chart_df[col] / 100.0

melted = chart_df.melt(id_vars=['Model'], value_vars=['Accuracy', 'Precision', 'Sensitivity', 'Specificity', 'F-measure'], var_name='Metric', value_name='Score')

plt.figure(figsize=(13, 5), dpi=150)
sns.barplot(data=melted, x='Model', y='Score', hue='Metric', palette='tab10')
plt.title("Performance Metric Comparison Across Models", fontsize=13, fontweight='bold', pad=12)
plt.ylim(0.70, 1.02)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Confusion Matrices for All 8 Models


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), dpi=150)
axes = axes.flatten()

for idx, (name, res) in enumerate(results_dict.items()):
    sns.heatmap(
        res['Confusion_Matrix'], annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
        xticklabels=['No HD', 'HD'], yticklabels=['No HD', 'HD'], annot_kws={'size': 12, 'fontweight': 'bold'}
    )
    axes[idx].set_title(f"{name} (Acc: {res['Accuracy']*100:.1f}%)", fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.show()
